In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

df_raw = pd.read_csv('Marine_Microplastics.csv')

cols_to_drop = ['GlobalID', 'Accession Link', 'Long Reference', 'DOI', 'x', 'y',
                'Accession Number', 'Measurement', 'Density Range', 'OBJECTID']
df = df_raw.drop(columns=[c for c in cols_to_drop if c in df_raw.columns])

if 'Date' in df_raw.columns:

    df['Date'] = pd.to_datetime(df_raw['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
    df['Year'] = df['Date'].dt.year.fillna(df['Date'].dt.year.median())
    df['Month'] = df['Date'].dt.month.fillna(6)
    df = df.drop(columns=['Date'])


density_map = {'Very Low': 0, 'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4}
df['Density Class'] = df['Density Class'].map(density_map)
df = df.dropna(subset=['Density Class'])

le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df['lat_sin'] = np.sin(np.radians(df['Latitude']))
df['lat_cos'] = np.cos(np.radians(df['Latitude']))
df['lon_sin'] = np.sin(np.radians(df['Longitude']))
df['lon_cos'] = np.cos(np.radians(df['Longitude']))

df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)

df = df.drop(columns=['Latitude', 'Longitude', 'Month'])

X = df.drop(columns=['Density Class'])
y = df['Density Class']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print(f"Data Pipeline Complete: {len(X_resampled)} balanced samples generated.")


X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("\n--- FINAL CLASSIFICATION PERFORMANCE ---")
print(classification_report(y_test, y_pred, target_names=list(density_map.keys())))

plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=density_map.keys(), yticklabels=density_map.keys())
plt.title('Marine Pollution Classifier: Balanced Confusion Matrix')
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.show()

feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 5))
feat_imp.head(10).plot(kind='bar', color='#1C7293')
plt.title('Key Environmental Drivers of Microplastic Density')
plt.show()